In [ ]:
# ============================================
# CELDA 1 — Instalar dependencias
# ============================================
!pip install -U pip -q
!pip install whisperx -q          # ya trae pyannote.audio como dependencia
!pip install "numpy==2.2.6" -q    # compatible con whisperx, pyannote y numba

In [ ]:
# ============================================
# CELDA 2 — Verificar GPU y numpy
# ============================================
import torch
import numpy
print("numpy:", numpy.__version__)
print("GPU disponible:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU ❌")

In [ ]:
# ============================================
# CELDA 3 — Servidor de transcripción remota
#   (async + diarización + idioma fijo + tiempos por palabra)
# ============================================
!pip install flask -q

import os, time, tempfile, threading, subprocess, uuid
import pandas as pd
from flask import Flask, request, jsonify
import whisperx, torch, torchaudio
from huggingface_hub import login
from pyannote.audio import Pipeline as DiarizationPipeline

from google.colab import userdata   # Secrets de Colab (icono de llave en la barra lateral)
HF_TOKEN   = userdata.get("HF_TOKEN")
def _secret(*nombres):
    """Primer Secret de Colab que exista y no esté vacío (cada cuenta puede usar otro nombre)."""
    for nombre in nombres:
        try:
            valor = userdata.get(nombre)
        except Exception:   # SecretNotFoundError / sin "Acceso del notebook"
            continue
        if valor:
            return valor
    return None

# Debe ser igual a COLAB_AUTH_TOKEN del .env; el Secret puede llamarse AUTH_TOKEN o COLAB_AUTH_TOKEN
AUTH_TOKEN = _secret("AUTH_TOKEN", "COLAB_AUTH_TOKEN")
if not AUTH_TOKEN:
    raise RuntimeError("Falta el Secret AUTH_TOKEN o COLAB_AUTH_TOKEN (🔑, con 'Acceso del notebook'): "
                       "sin clave el servidor quedaría abierto a cualquiera")

MODEL_NAME       = "medium"   # confirmado: la segmentación no dependía del modelo, medium es seguro
IDIOMA_DEFECTO   = "es"       # si el pipeline no manda idioma, se usa este (no se adivina)

login(token=HF_TOKEN)

print(f"Cargando modelo '{MODEL_NAME}' en GPU (una sola vez)...")
device = "cuda"
model = whisperx.load_model(MODEL_NAME, device=device, compute_type="float16")

# Un modelo de alineación por idioma, cargado a demanda (español ya precargado)
align_models = {IDIOMA_DEFECTO: whisperx.load_align_model(language_code=IDIOMA_DEFECTO, device=device)}
align_lock = threading.Lock()

def _align_model(lang):
    with align_lock:
        if lang not in align_models:
            print(f"[SERVER] Cargando modelo de alineación para '{lang}'...")
            align_models[lang] = whisperx.load_align_model(language_code=lang, device=device)
        return align_models[lang]

diarize_model = DiarizationPipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1", token=HF_TOKEN
).to(torch.device(device))
print("✅ Modelos listos — servidor arrancando...")

app = Flask(__name__)

jobs = {}
jobs_lock = threading.Lock()

def _process_job(job_id, tmp_path, language):
    try:
        t0 = time.time()
        audio  = whisperx.load_audio(tmp_path)

        # Idioma fijo: sin esto WhisperX lo adivina con los primeros segundos (y puede
        # equivocarse, ej. transcribir español como italiano)
        result = model.transcribe(audio, batch_size=16, language=language)
        idioma = result.get("language") or language   # guardarlo ANTES de align (align lo pierde)

        model_a, metadata = _align_model(idioma)
        result = whisperx.align(result["segments"], model_a, metadata, audio, device,
                                return_char_alignments=False)

        waveform, sample_rate = torchaudio.load(tmp_path)
        raw_output = diarize_model({"waveform": waveform, "sample_rate": sample_rate})

        # ── Fix confirmado: convertir el contenedor de pyannote 4.x a DataFrame ──
        diarization = raw_output.speaker_diarization if hasattr(raw_output, "speaker_diarization") else raw_output
        diarize_df  = pd.DataFrame(diarization.itertracks(yield_label=True), columns=['segment', 'label', 'speaker'])
        diarize_df['start'] = diarize_df['segment'].apply(lambda x: x.start)
        diarize_df['end']   = diarize_df['segment'].apply(lambda x: x.end)

        try:
            result = whisperx.assign_word_speakers(diarize_df, result)
            print(f"[SERVER] Job {job_id}: diarización OK ({len(diarize_df)} tracks)")
        except Exception as e:
            print(f"[SERVER] Job {job_id}: ⚠ assign_word_speakers falló: {type(e).__name__}: {e}")
        # ──────────────────────────────────────────────────────────────────────

        segments_out = [
            {
                "start":   float(seg.get("start", 0)),
                "text":    seg.get("text", "").strip(),
                "speaker": seg.get("speaker"),
                # Tiempos por palabra: el pipeline los usa para los subtítulos animados
                "words": [
                    {"word": w["word"], "start": float(w["start"]), "end": float(w["end"])}
                    for w in seg.get("words", []) if "start" in w and "end" in w
                ],
            }
            for seg in result["segments"] if seg.get("text", "").strip()
        ]
        proc_time = round(time.time() - t0, 1)
        print(f"[SERVER] Job {job_id}: {len(segments_out)} segmentos en {proc_time}s (idioma: {idioma})")
        with jobs_lock:
            jobs[job_id] = {
                "status": "done",
                "segments": segments_out,
                "language": idioma,          # el idioma real con el que se transcribió
                "processing_seconds": proc_time,
            }
    except Exception as e:
        print(f"[SERVER] Job {job_id} falló: {e}")
        with jobs_lock:
            jobs[job_id] = {"status": "error", "error": str(e)}
    finally:
        os.remove(tmp_path)

@app.route("/transcribe", methods=["POST"])
def transcribe():
    if AUTH_TOKEN and request.headers.get("Authorization") != f"Bearer {AUTH_TOKEN}":
        return jsonify({"error": "unauthorized"}), 401

    audio_file = request.files.get("audio")
    if not audio_file:
        return jsonify({"error": "falta el archivo 'audio'"}), 400

    # El pipeline manda el idioma del nicho; si no viene, se usa el de por defecto
    language = (request.form.get("language") or IDIOMA_DEFECTO).strip().lower()

    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
        audio_file.save(tmp.name)
        tmp_path = tmp.name

    job_id = uuid.uuid4().hex
    with jobs_lock:
        jobs[job_id] = {"status": "processing"}
    threading.Thread(target=_process_job, args=(job_id, tmp_path, language), daemon=True).start()
    print(f"[SERVER] Job {job_id} recibido (idioma: {language}), procesando en segundo plano...")
    return jsonify({"job_id": job_id}), 202

@app.route("/status/<job_id>", methods=["GET"])
def status(job_id):
    if AUTH_TOKEN and request.headers.get("Authorization") != f"Bearer {AUTH_TOKEN}":
        return jsonify({"error": "unauthorized"}), 401
    with jobs_lock:
        job = jobs.get(job_id)
    if job is None:
        return jsonify({"error": "job_id no encontrado"}), 404
    return jsonify(job)

threading.Thread(target=lambda: app.run(port=5000), daemon=True).start()
time.sleep(2)

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Esperando URL pública...")
for line in proc.stdout:
    print(line, end="")
    if "https://" in line and "trycloudflare.com" in line:
        break